# 05 — Estimate productivity trajectory

Estimate the direction, rate, and statistical evidence of change in annual NDVI using Mann–Kendall Z and Sen's slope.


## Inputs, outputs, and execution order

**Inputs:** Annual rasters from 04; the defaults require a complete 16-year sequence for each reporting window.

**Outputs:** Mann–Kendall Z, Sen slope, optional OLS and p-value rasters, class maps, and whole-area trend PNGs under `trajectory/`.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

Sen slope is NDVI units per year. Mann–Kendall Z is a standardized test statistic, not Kendall's tau. The binary trajectory flags Z < −1.96. The inherited implementation uses the untied, independent-observation variance and a continuity correction; it does not correct ties, serial correlation, multiple testing, or climatic drivers. Keep complete-pixel time series enabled. The whole-area trend of spatial medians differs from the spatial distribution of pixel trends.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
%matplotlib inline

import os
import re
import gc
import glob
import math
import warnings

import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import Window

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Choose reporting windows and outputs

Defaults give 2000–2015, 2004–2019, and 2008–2023 for a 2000–2024 series. The trailing year 2024 is outside these four-year reporting steps. Optional outputs can be disabled; existing files are reused only when all requested products exist.


In [ ]:
# =========================================================
# USER-DEFINED PARAMETERS
# =========================================================
TRENDS_OUTPUT = cfg["output_dir"]
ANNUAL_PATTERN = f"{TRENDS_OUTPUT}/annual_ndvi/ndvi_annual_{cfg['annual_metric']}_*.tif"
INDICATOR_OUTPUT = f"{TRENDS_OUTPUT}/trajectory"

WINDOW_YEARS = cfg["trajectory_window_years"]
WINDOW_STEP = cfg["report_step_years"]
BLOCKSIZE = cfg["block_size"]

ENFORCE_FULL_COMPLETENESS = True
MIN_VALID_YEARS = None   # e.g. 12 if not enforcing full completeness

WRITE_OLS = True
WRITE_P = True
WRITE_SIG_P05 = True
WRITE_SIG_DIR = True
WRITE_Z5 = False
WRITE_LDN3 = True
WRITE_SDG2 = True

SKIP_EXISTING = cfg["skip_existing"]

MAX_SAMPLES_FOR_WHOLE_MAP_MEDIAN = 2_000_000
WHOLE_MAP_RANDOM_SEED = 0


## 4. Discover annual rasters and verify their grids

All years in a reporting window must be present and spatially aligned. The cleanup helper requests garbage collection; it does not delete arrays held by the caller.


In [ ]:
# =========================================================
# HELPER: MEMORY CLEANUP
# =========================================================
def clear_memory(*objs):
    for obj in objs:
        obj = None
    gc.collect()


# =========================================================
# HELPER: PARSE YEAR / YYYYMM FROM FILE NAMES
# =========================================================
def year_from_path(path):
    base = os.path.basename(path)
    m = re.search(r"(19|20)\d{2}", base)
    if not m:
        raise ValueError(f"Could not parse year from filename: {base}")
    return int(m.group(0))


# =========================================================
# HELPER: CREATE OUTPUT DIRECTORY
# =========================================================
def ensure_parent_dir(path):
    if path:
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)


# =========================================================
# HELPER: SELECT 16-YEAR SLIDING WINDOWS
# =========================================================
def sliding_windows_step(annual_pattern, window_years=16, step=4):
    paths = sorted(glob.glob(annual_pattern))
    if not paths:
        raise ValueError(f"No annual rasters found for pattern: {annual_pattern}")

    years = np.array([year_from_path(p) for p in paths])
    if len(set(years)) != len(years):
        raise ValueError("Duplicate annual years; select one annual metric/version")
    year_to_path = dict(zip(years, paths))

    first_year = years.min()
    last_year = years.max()

    windows = []
    start = first_year

    while start + window_years - 1 <= last_year:
        end = start + window_years - 1
        window_years_list = list(range(start, end + 1))

        if all(y in year_to_path for y in window_years_list):
            window_paths = [year_to_path[y] for y in window_years_list]
            windows.append((start, end, window_paths))

        start += step

    return windows


# =========================================================
# HELPER: CHECK ALL INPUT RASTERS SHARE SAME GRID
# =========================================================
def assert_same_grid(paths):
    with rasterio.open(paths[0]) as ref:
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height

    for p in paths[1:]:
        with rasterio.open(p) as src:
            if src.crs != ref_crs:
                raise ValueError(f"CRS mismatch in {p}: {src.crs} != {ref_crs}")
            if src.transform != ref_transform:
                raise ValueError(f"Transform mismatch in {p}")
            if src.width != ref_width or src.height != ref_height:
                raise ValueError(
                    f"Shape mismatch in {p}: {(src.width, src.height)} != {(ref_width, ref_height)}"
                )


# =========================================================
# HELPER: CONVERT NODATA TO NAN
# =========================================================
def nodata_to_nan(arr, nodata):
    arr = arr.astype("float32", copy=False)

    if nodata is None:
        return arr

    if isinstance(nodata, float) and np.isnan(nodata):
        return arr

    return np.where(arr == nodata, np.nan, arr)


# =========================================================
# HELPER: MAKE OUTPUT PROFILES
# =========================================================
def build_output_profiles(ref_profile, width, height, blocksize):
    float_profile = ref_profile.copy()
    float_profile.update(
        dtype="float32",
        count=1,
        nodata=np.nan,
        compress="deflate",
        tiled=True,
        blockxsize=blocksize,
        blockysize=blocksize,
    )

    u8_profile = ref_profile.copy()
    u8_profile.update(
        dtype="uint8",
        count=1,
        nodata=255,
        compress="deflate",
        tiled=True,
        blockxsize=blocksize,
        blockysize=blocksize,
    )

    i8_profile = ref_profile.copy()
    i8_profile.update(
        dtype="int8",
        count=1,
        nodata=-128,
        compress="deflate",
        tiled=True,
        blockxsize=blocksize,
        blockysize=blocksize,
    )

    sdg_profile = ref_profile.copy()
    sdg_profile.update(
        dtype="uint8",
        count=1,
        nodata=255,
        compress="deflate",
        tiled=True,
        blockxsize=blocksize,
        blockysize=blocksize,
    )

    return float_profile, u8_profile, i8_profile, sdg_profile


## 5. Define trend statistics

Pairwise signs produce Mann–Kendall S and a standardized Z. The median of all pairwise slopes gives Sen's slope. Default inference assumes complete, independent, untied observations.


In [ ]:
# =========================================================
# MANN-KENDALL Z + SEN SLOPE FOR A BLOCK
# =========================================================
def mk_z_and_sens_slope_block(Y, years=None):
    Y = Y.astype("float32", copy=False)
    T, H, W = Y.shape

    t = np.arange(T, dtype="float32") if years is None else np.asarray(years, dtype="float32")

    finite = np.isfinite(Y)
    n_obs = finite.sum(axis=0)
    valid = n_obs >= 2

    Z = np.full((H, W), np.nan, dtype="float32")
    sen_slope = np.full((H, W), np.nan, dtype="float32")

    if not np.any(valid):
        return Z, sen_slope

    S = np.zeros((H, W), dtype="int32")

    for i in range(T - 1):
        yi = Y[i]
        for j in range(i + 1, T):
            diff = Y[j] - yi
            sgn = np.sign(diff)
            sgn = np.where(np.isfinite(sgn), sgn, 0).astype("int8")
            S += sgn

    n = T
    varS = (n * (n - 1) * (2 * n + 5)) / 18.0
    sqrt_varS = np.sqrt(varS).astype("float32")

    S_f = S.astype("float32")
    Z_all = np.zeros((H, W), dtype="float32")
    Z_all = np.where(S_f > 0, (S_f - 1.0) / sqrt_varS, Z_all)
    Z_all = np.where(S_f < 0, (S_f + 1.0) / sqrt_varS, Z_all)
    Z[valid] = Z_all[valid]

    slopes = []
    for i in range(T - 1):
        yi = Y[i]
        ti = t[i]
        for j in range(i + 1, T):
            dt = t[j] - ti
            slopes.append((Y[j] - yi) / dt)

    slopes = np.stack(slopes, axis=0)
    slope_all = np.nanmedian(slopes, axis=0).astype("float32")
    sen_slope[valid] = slope_all[valid]

    return Z, sen_slope


# =========================================================
# OLS SLOPE FOR A BLOCK
# =========================================================
def ols_slope_block(Y, years=None):
    Y = Y.astype("float32", copy=False)
    T, H, W = Y.shape

    x = np.arange(T, dtype="float32") if years is None else np.asarray(years, dtype="float32")
    X = x[:, None, None]

    m = np.isfinite(Y)
    n = m.sum(axis=0)
    ok = n >= 2

    slope = np.full((H, W), np.nan, dtype="float32")
    if not np.any(ok):
        return slope

    sum_x = np.nansum(np.where(m, X, np.nan), axis=0)
    sum_y = np.nansum(np.where(m, Y, np.nan), axis=0)

    mean_x = sum_x / n
    mean_y = sum_y / n

    num = np.nansum(np.where(m, (X - mean_x) * (Y - mean_y), np.nan), axis=0)
    den = np.nansum(np.where(m, (X - mean_x) ** 2, np.nan), axis=0)

    out = num / den
    slope[ok] = out[ok].astype("float32")

    return slope


# =========================================================
# P-VALUE FROM Z
# =========================================================
def mk_p_from_z(Z):
    Z = Z.astype("float32")
    Za = np.abs(Z)

    erf_vec = np.vectorize(math.erf)
    cdf = 0.5 * (1.0 + erf_vec(Za / np.sqrt(2.0)))
    p = 2.0 * (1.0 - cdf)

    p[~np.isfinite(Z)] = np.nan
    return p.astype("float32")


## 6. Define trajectory classes and spatial summaries

Three classes: 1 decreasing, 2 stable, 3 increasing at ±1.28. Five classes split at −1.96, −1.28, 1.28, 1.96. Binary SDG2-named output: 1 for Z < −1.96, 0 otherwise, 255 nodata. This filename does not establish full SDG 15.3.1 compliance. The spatial-median helper concatenates raster values before sampling, so its sample cap does not cap peak RAM.


In [ ]:
# =========================================================
# CLASSIFICATION HELPERS
# =========================================================
def classify_z_5level(Z):
    out = np.full(Z.shape, 255, dtype="uint8")
    out[np.isfinite(Z) & (Z < -1.96)] = 1
    out[np.isfinite(Z) & (Z >= -1.96) & (Z < -1.28)] = 2
    out[np.isfinite(Z) & (Z >= -1.28) & (Z <= 1.28)] = 3
    out[np.isfinite(Z) & (Z > 1.28) & (Z <= 1.96)] = 4
    out[np.isfinite(Z) & (Z > 1.96)] = 5
    return out


def classify_ldn_3class(Z):
    out = np.full(Z.shape, 255, dtype="uint8")
    out[np.isfinite(Z) & (Z < -1.28)] = 1
    out[np.isfinite(Z) & (Z >= -1.28) & (Z <= 1.28)] = 2
    out[np.isfinite(Z) & (Z > 1.28)] = 3
    return out


def classify_sdg2(Z):
    out = np.full(Z.shape, 255, dtype="uint8")  # nodata
    valid = np.isfinite(Z)
    out[valid] = 0
    out[valid & (Z < -1.96)] = 1
    return out


# =========================================================
# WHOLE-MAP MEDIAN FROM RASTER
# =========================================================
def spatial_median_raster(path, max_samples=2_000_000, seed=0):
    rng = np.random.default_rng(seed)
    samples = []

    with rasterio.open(path) as src:
        nd = src.nodata

        for _, win in src.block_windows(1):
            a = src.read(1, window=win).astype("float32")
            a = nodata_to_nan(a, nd)

            vals = a[np.isfinite(a)]
            if vals.size:
                samples.append(vals)

    if not samples:
        return np.nan

    vals = np.concatenate(samples)

    if vals.size > max_samples:
        idx = rng.choice(vals.size, size=max_samples, replace=False)
        vals = vals[idx]

    return float(np.median(vals))


# =========================================================
# WHOLE-MAP MK Z AND SEN SLOPE
# =========================================================
def mann_kendall_z_1d(y):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    n = len(y)

    if n < 2:
        return np.nan

    S = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            if y[j] > y[i]:
                S += 1
            elif y[j] < y[i]:
                S -= 1

    varS = n * (n - 1) * (2 * n + 5) / 18.0

    if S > 0:
        return (S - 1) / np.sqrt(varS)
    elif S < 0:
        return (S + 1) / np.sqrt(varS)
    else:
        return 0.0


def sens_slope_1d(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)

    slopes = []
    for i in range(len(y) - 1):
        for j in range(i + 1, len(y)):
            if np.isfinite(y[i]) and np.isfinite(y[j]) and (x[j] != x[i]):
                slopes.append((y[j] - y[i]) / (x[j] - x[i]))

    if not slopes:
        return np.nan

    return float(np.median(slopes))


# =========================================================
# PLOT WHOLE-MAP TREND
# =========================================================
def plot_whole_map_trend(years, med_ndvi, Z, slope, title, out_png=None):
    intercept = np.median(med_ndvi - slope * years)

    plt.figure(figsize=(8, 5))
    plt.scatter(years, med_ndvi, color='red', label='Median NDVI')
    plt.plot(years, intercept + slope * years, color='black', linewidth=2, label='Trend')
    
    plt.legend()

    if abs(Z) >= 1.96:
        interpretation = "Significant increase" if slope > 0 else "Significant decrease"
    else:
        interpretation = "No Significant Change"

    txt = f"Slope = {slope:.5f}\nMann-Kendall Z = {Z:.3f} ({interpretation})"
    plt.text(years.min(), np.nanmax(med_ndvi), txt, fontsize=11, va="top")

    plt.xlabel("Year")
    plt.ylabel("Median NDVI")
    plt.title(title)
    plt.grid(True, alpha=0.3)

    if out_png:
        ensure_parent_dir(out_png)
        plt.savefig(out_png, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close()


## 7. Process a reporting window

Spatial blocks limit the pixel stack. Smaller blocks reduce pairwise-slope memory use. Valid zero significance and direction values remain distinct from nodata.


In [ ]:
# =========================================================
# MAIN BLOCK-WISE TREND COMPUTATION FOR ONE WINDOW
# =========================================================
def compute_trend_window(
    annual_paths,
    out_z_tif,
    out_sen_slope_tif,
    out_ols_slope_tif=None,
    out_p_tif=None,
    out_sig_p05_tif=None,
    out_sig_dir_tif=None,
    out_z5_tif=None,
    out_ldn3_tif=None,
    out_sdg2_tif=None,
    enforce_full_completeness=True,
    min_valid_years=None,
    blocksize=512,
    skip_existing=True,
):
    years = [year_from_path(p) for p in annual_paths]

    for fp in [
        out_z_tif, out_sen_slope_tif, out_ols_slope_tif, out_p_tif,
        out_sig_p05_tif, out_sig_dir_tif, out_z5_tif, out_ldn3_tif, out_sdg2_tif
    ]:
        ensure_parent_dir(fp)

    requested_paths = [p for p in [out_z_tif, out_sen_slope_tif, out_ols_slope_tif,
        out_p_tif, out_sig_p05_tif, out_sig_dir_tif, out_z5_tif, out_ldn3_tif, out_sdg2_tif] if p]
    if skip_existing and all(os.path.exists(p) for p in requested_paths):
        print(f"Skipping existing outputs for window {years[0]}-{years[-1]}")
        return {
            "years": years,
            "z": out_z_tif,
            "sen_slope": out_sen_slope_tif,
            "ols_slope": out_ols_slope_tif,
            "p": out_p_tif,
            "sig_p05": out_sig_p05_tif,
            "sig_dir": out_sig_dir_tif,
            "z5": out_z5_tif,
            "ldn3": out_ldn3_tif,
            "sdg2": out_sdg2_tif,
        }

    assert_same_grid(annual_paths)

    with rasterio.open(annual_paths[0]) as src0:
        profile = src0.profile.copy()
        H, W = src0.height, src0.width

    float_profile, u8_profile, i8_profile, sdg_profile = build_output_profiles(
        profile, W, H, blocksize
    )

    srcs = [rasterio.open(p) for p in annual_paths]

    dst_ols = None
    dst_p = None
    dst_sig = None
    dst_dir = None
    dst_z5 = None
    dst_ldn = None
    dst_sdg = None

    try:
        with rasterio.open(out_z_tif, "w", **float_profile) as dst_z, \
             rasterio.open(out_sen_slope_tif, "w", **float_profile) as dst_sen:

            dst_z.update_tags(Trend="Mann-Kendall Z", Years=",".join(map(str, years)))
            dst_sen.update_tags(Trend="Sen slope (per year)", Years=",".join(map(str, years)))

            if out_ols_slope_tif:
                dst_ols = rasterio.open(out_ols_slope_tif, "w", **float_profile)
                dst_ols.update_tags(Trend="OLS slope", Years=",".join(map(str, years)))

            if out_p_tif:
                dst_p = rasterio.open(out_p_tif, "w", **float_profile)
                dst_p.update_tags(Test="Mann-Kendall p-value", TwoSided="True")

            if out_sig_p05_tif:
                dst_sig = rasterio.open(out_sig_p05_tif, "w", **u8_profile)
                dst_sig.update_tags(Significance="p<=0.05")

            if out_sig_dir_tif:
                dst_dir = rasterio.open(out_sig_dir_tif, "w", **i8_profile)
                dst_dir.update_tags(Class="-1 sig dec, 0 not sig, +1 sig inc")

            if out_z5_tif:
                dst_z5 = rasterio.open(out_z5_tif, "w", **u8_profile)

            if out_ldn3_tif:
                dst_ldn = rasterio.open(out_ldn3_tif, "w", **u8_profile)

            if out_sdg2_tif:
                dst_sdg = rasterio.open(out_sdg2_tif, "w", **sdg_profile)

            for row0 in range(0, H, blocksize):
                for col0 in range(0, W, blocksize):
                    h = min(blocksize, H - row0)
                    w = min(blocksize, W - col0)
                    win = Window(col0, row0, w, h)

                    stack = []
                    for src in srcs:
                        a = src.read(1, window=win)
                        a = nodata_to_nan(a, src.nodata)
                        stack.append(a)

                    Y = np.stack(stack, axis=0)  # (T, h, w)

                    if enforce_full_completeness:
                        ok = np.isfinite(Y).all(axis=0)
                        Y = np.where(ok[None, :, :], Y, np.nan)
                    elif min_valid_years is not None:
                        ok = np.isfinite(Y).sum(axis=0) >= int(min_valid_years)
                        Y = np.where(ok[None, :, :], Y, np.nan)

                    Z, sen_slope = mk_z_and_sens_slope_block(Y, years=years)
                    dst_z.write(Z, 1, window=win)
                    dst_sen.write(sen_slope, 1, window=win)

                    if dst_ols:
                        ols = ols_slope_block(Y, years=years)
                        dst_ols.write(ols, 1, window=win)
                    else:
                        ols = None

                    direction = None  # Also defined when only p-values are requested.
                    if dst_p or dst_sig or dst_dir:
                        pvals = mk_p_from_z(Z)

                        if dst_p:
                            dst_p.write(pvals, 1, window=win)

                        sig = np.isfinite(pvals) & (pvals <= 0.05)

                        if dst_sig:
                            dst_sig.write(np.where(np.isfinite(Z), sig, 255).astype("uint8"), 1, window=win)

                        if dst_dir:
                            direction = np.zeros(Z.shape, dtype="int8")
                            direction[sig & (sen_slope > 0)] = 1
                            direction[sig & (sen_slope < 0)] = -1
                            direction[~np.isfinite(Z)] = -128
                            dst_dir.write(direction, 1, window=win)
                    else:
                        pvals = None
                        sig = None
                        direction = None

                    if dst_z5:
                        dst_z5.write(classify_z_5level(Z), 1, window=win)

                    if dst_ldn:
                        dst_ldn.write(classify_ldn_3class(Z), 1, window=win)

                    if dst_sdg:
                        dst_sdg.write(classify_sdg2(Z), 1, window=win)

                    clear_memory(stack, Y, Z, sen_slope, ols, pvals, sig, direction)

        return {
            "years": years,
            "z": out_z_tif,
            "sen_slope": out_sen_slope_tif,
            "ols_slope": out_ols_slope_tif,
            "p": out_p_tif,
            "sig_p05": out_sig_p05_tif,
            "sig_dir": out_sig_dir_tif,
            "z5": out_z5_tif,
            "ldn3": out_ldn3_tif,
            "sdg2": out_sdg2_tif,
        }

    finally:
        for src in srcs:
            try:
                src.close()
            except Exception:
                pass

        for dst in [dst_ols, dst_p, dst_sig, dst_dir, dst_z5, dst_ldn, dst_sdg]:
            try:
                if dst is not None:
                    dst.close()
            except Exception:
                pass

        clear_memory(srcs, dst_ols, dst_p, dst_sig, dst_dir, dst_z5, dst_ldn, dst_sdg)


## 8. Summarize annual maps

The whole-area curve is diagnostic and can mask local decreases. A fixed seed makes the subsampled spatial median repeatable for a fixed input series.


In [ ]:
# =========================================================
# PRECOMPUTE WHOLE-MAP MEDIAN SERIES ONCE
# =========================================================
def build_whole_map_median_series(annual_pattern, max_samples=2_000_000, seed=0):
    annual_paths = sorted(glob.glob(annual_pattern))
    if not annual_paths:
        raise ValueError(f"No annual rasters found for: {annual_pattern}")

    years = np.array([year_from_path(p) for p in annual_paths])
    medians = np.array([
        spatial_median_raster(p, max_samples=max_samples, seed=seed)
        for p in annual_paths
    ])

    return years, medians, annual_paths


## 9. Run trajectory and whole-area plots

Check the printed year windows and file list. Use the SDG2 binary raster for integration in notebook 08, not the signed direction raster.


In [ ]:
# =========================================================
# MAIN EXECUTION
# =========================================================
windows = sliding_windows_step(
    ANNUAL_PATTERN,
    window_years=WINDOW_YEARS,
    step=WINDOW_STEP,
)

if not windows:
    raise ValueError("No complete reporting windows. Generate more annual rasters or review the configured period.")

print("Detected reporting windows:")
for start, end, paths in windows:
    print(f"  {start}-{end} | {len(paths)} rasters")

years_all, med_series_all, annual_paths_all = build_whole_map_median_series(
    ANNUAL_PATTERN,
    max_samples=MAX_SAMPLES_FOR_WHOLE_MAP_MEDIAN,
    seed=WHOLE_MAP_RANDOM_SEED,
)

for start, end, annual_paths in windows:
    print("\n" + "=" * 70)
    print(f"Processing window {start}-{end}")
    print("=" * 70)

    outs = compute_trend_window(
        annual_paths=annual_paths,
        out_z_tif=f"{INDICATOR_OUTPUT}/mkZ_{start}_{end}.tif",
        out_sen_slope_tif=f"{INDICATOR_OUTPUT}/sen_slope_{WINDOW_YEARS}yr_{start}_{end}.tif",
        out_ols_slope_tif=f"{INDICATOR_OUTPUT}/ols_slope_{WINDOW_YEARS}yr_{start}_{end}.tif" if WRITE_OLS else None,
        out_p_tif=f"{INDICATOR_OUTPUT}/mk_p_{WINDOW_YEARS}yr_{start}_{end}.tif" if WRITE_P else None,
        out_sig_p05_tif=f"{INDICATOR_OUTPUT}/sig_p05_{WINDOW_YEARS}yr_{start}_{end}.tif" if WRITE_SIG_P05 else None,
        out_sig_dir_tif=f"{INDICATOR_OUTPUT}/sig_dir_{WINDOW_YEARS}yr_{start}_{end}.tif" if WRITE_SIG_DIR else None,
        out_z5_tif=f"{INDICATOR_OUTPUT}/Z5_{WINDOW_YEARS}yr_{start}_{end}.tif" if WRITE_Z5 else None,
        out_ldn3_tif=f"{INDICATOR_OUTPUT}/LDN3_{WINDOW_YEARS}yr_{start}_{end}.tif" if WRITE_LDN3 else None,
        out_sdg2_tif=f"{INDICATOR_OUTPUT}/SDG2_{WINDOW_YEARS}yr_{start}_{end}.tif" if WRITE_SDG2 else None,
        enforce_full_completeness=ENFORCE_FULL_COMPLETENESS,
        min_valid_years=MIN_VALID_YEARS,
        blocksize=BLOCKSIZE,
        skip_existing=SKIP_EXISTING,
    )

    print("Years used:", outs["years"])
    print("Outputs written:", outs)

    # Whole-map median trend for the same window
    mask = (years_all >= start) & (years_all <= end)
    years_win = years_all[mask]
    med_win = med_series_all[mask]

    Z_whole = mann_kendall_z_1d(med_win)
    slope_whole = sens_slope_1d(med_win, years_win)

    if Z_whole < -1.96:
        interpretation = "Significant decrease (degraded)"
    elif Z_whole < -1.28:
        interpretation = "Potentially degrading"
    elif Z_whole <= 1.28:
        interpretation = "Stable"
    elif Z_whole <= 1.96:
        interpretation = "Potentially improving"
    else:
        interpretation = "Significant increase (improving)"

    print(f"Whole-map MK Z: {Z_whole:.3f}")
    print(f"Whole-map Sen slope: {slope_whole:.5f}")
    print(f"Interpretation: {interpretation}")

    plot_whole_map_trend(
        years=years_win,
        med_ndvi=med_win,
        Z=Z_whole,
        slope=slope_whole,
        title=f"Productivity Trajectory {start}-{end}",
        out_png=f"{INDICATOR_OUTPUT}/whole_map_trend_{start}-{end}.png",
    )

    clear_memory(mask, years_win, med_win, Z_whole, slope_whole, outs)

print("\n✓ All windows processed.")


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
